#**Module 2-End Assignment - Online retail dataset**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

###**Task 1 – Data Import & Setup**

In [ ]:
#Load dataset using Pandas
df = pd.read_excel("https://raw.githubusercontent.com/Chenneumakanth/Online-retail-dataset/refs/heads/main/Online%20Retail.xlsx")

In [ ]:
#Head - First 5 rows
df.head()

In [ ]:
#tail - last 5 rows
df.tail()

In [ ]:
#Shape of the dataset
df.shape

In [ ]:
#Cloumns in the dataset
df.columns

In [ ]:
#dtypes of the column
df.dtypes

In [ ]:
#To check null values in dataset
df.isnull().sum()

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'],errors="coerce")

In [ ]:
df.head()

###**Task 2 – Data Cleaning**

In [ ]:
#Handle missing values (remove null CustomerID)
df.drop(df[df['CustomerID'].isnull()].index,axis = 0,inplace= True)

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
#Remove duplicates
df.drop(df[df.duplicated].index,axis=0,inplace = True)

In [ ]:
df.duplicated().sum()

In [ ]:
#Fix invalid values (negative quantity, invalid price)
quantity_mean_price = df["Quantity"].mean()
df.loc[df["Quantity"] <= 0, "Quantity"] = quantity_mean_price
unit_mean_price = df["UnitPrice"].mean()
df.loc[df["UnitPrice"] <= 0, "UnitPrice"] = unit_mean_price

###**Task 3 – Feature Engineering**

In [ ]:
#Create TotalPrice = Quantity × UnitPrice
df["TotalPrice"] = df['Quantity'] * df['UnitPrice']

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
#Extract time features (Year, Month, Day, Hour)
df['Date'] = df['InvoiceDate'].dt.date
df['Month'] = df['InvoiceDate'].dt.month_name()
df['Year'] = df['InvoiceDate'].dt.year
df['Hour'] = df['InvoiceDate'].dt.hour
df['Day'] = df['InvoiceDate'].dt.day

In [ ]:
df.tail()


In [ ]:
# Create categories (Customer Segment)
customer_segments = (
    df.groupby("Country")
    .agg(TotalRevenue=("TotalPrice", "sum"), TotalOrders=("InvoiceNo", "nunique"))
    .reset_index()
)
customer_segments["MarketSegment"] = pd.cut(
    customer_segments["TotalRevenue"],
    bins=[-np.inf, 1000, 10000, np.inf],
    labels=["Emerging Market", "Developing Market", "Core Market"]
)
df["CustomerSegment"] = df["Country"].map(
    customer_segments.set_index("Country")["MarketSegment"]
)

In [ ]:
# Create categories (Day Type)
df["DayName"] = df['InvoiceDate'].dt.day_name()
df["DayType"] = df['DayName'].apply(lambda x: "Weekend" if x in ["Saturday","Sunday"] else "Weekday")

In [ ]:
# Create categories (Order Size)
df["OrderSize"] = pd.cut(
    df['Quantity'],
    bins=[-np.inf, 100, 500, np.inf],
    labels=["Small Order", "Medium Order", "Bulk Order"])

In [ ]:
df.head()

###**Task 4 – Data Exploration**

#####**Use describe() and dataset overview**

In [ ]:
df.describe(include="number")

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
df.dtypes

#####**Analyze categories (value_counts(), unique())**

In [ ]:
#Value counts of Order size
df['OrderSize'].value_counts()

In [ ]:
#Value counts of day type
df['DayType'].value_counts()

In [ ]:
#Value counts of Customer Segment
df["CustomerSegment"].value_counts()

In [ ]:
#Unique values of Customer Segment
df["CustomerSegment"].unique()

In [ ]:
#Unique values of Order size
df['OrderSize'].unique()

In [ ]:
#Unique value of Day type
df['DayType'].unique()

#####**Perform groupby() (country, month, product)**

In [ ]:
#Group by based on country
df.groupby("Country")["TotalPrice"].sum().round(2)

In [ ]:
#Group by based on month
import warnings
warnings.filterwarnings("ignore")
month_order = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]
df["Month"] = pd.Categorical(df["Month"], categories=month_order, ordered=True)
df.groupby("Month")["Quantity"].sum()

In [ ]:
#Group by based on product
df.groupby("Description")["InvoiceNo"].count()

###**Task 5 – Data Wrangling**

In [ ]:
#Aggregate data using groupby()
df1 = df.groupby("Country").agg(
    Customer = ("InvoiceNo","unique"),
    TotalRevenue = ("TotalPrice","sum")
)
df1[["TotalRevenue"]].head(10)

In [ ]:
#Sort to find the top customers and countries
top_customers = (
    df.groupby("CustomerID").
    agg(TotalPrice = ("TotalPrice","sum")).
    sort_values(by="TotalPrice",ascending =  False).
    reset_index()
    )
top_customers['TotalPrice'] = top_customers['TotalPrice'].round(2)

In [ ]:
#Top Customers
top_customers.head(10)

In [ ]:
#Top_countries
top_countries = (
    df.groupby("Country").
    agg(TotalPrice = ("TotalPrice","sum")).
    sort_values(by="TotalPrice",ascending =  False).
    reset_index()
    )
top_countries['TotalPrice'] = top_countries['TotalPrice'].round(2)

In [ ]:
# Top Countries by sales
top_countries

In [ ]:
top_5_country_names = top_countries.head(5)["Country"].tolist()
filtered_df = df[df["Country"].isin(top_5_country_names)]

customer_country_pivot = filtered_df.pivot_table(
    values="TotalPrice",
    index="CustomerID",
    columns="Country",
    aggfunc="sum",
    fill_value=0
).round(2)
customer_country_pivot.head()

###**Task 6 – Statistical Analysis**

In [ ]:
#Analyze Quantity, UnitPrice, TotalPrice
anlyse_columns = ["Quantity", "UnitPrice", "TotalPrice"]

In [ ]:
df[anlyse_columns].describe()

In [ ]:
#Calculate mean, median,mode.standard deviation, variance, and percentiles
for col in anlyse_columns:
  col_mean = df[col].mean()
  col_median = df[col].median()
  col_mode = df[col].mode()[0]
  col_sd = df[col].std()
  col_variance = df[col].var()
  col_25_percentile = df[col].quantile(0.25)
  col_50_percentile = df[col].quantile(0.50)
  col_75_percentile = df[col].quantile(0.75)
  print(f'Mean for {col}: {col_mean:.2f}')
  print(f'Median for {col}: {col_median:.2f}')
  print(f'Mode for {col}: {col_mode}')
  print(f'Standard Deviation for {col}: {col_sd:.2f}')
  print(f'Variance for {col}: {col_variance:.2f}')
  print(f'25th percentile for {col}: {col_25_percentile:.2f}')
  print(f'50th percentile for {col}: {col_50_percentile:.2f}')
  print(f'75th percentile for {col}: {col_75_percentile:.2f}')

###**Task 7 – Data Visualization**
###**Matplotlib:**

In [ ]:
#Line chart
monthly_sales = df.groupby("Month")["TotalPrice"].sum().reset_index()
plt.figure(figsize=(10, 5))
plt.plot(monthly_sales["Month"],monthly_sales["TotalPrice"],marker='o')
plt.ticklabel_format(style="plain", axis="y")
plt.title("Sales by Month")
plt.xlabel("Month")
plt.ylabel("Total Sales Revenue ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
#Bar chart
month_spend_average = df.groupby("Month").agg(
                               TotalRevenue = ("TotalPrice","mean")
                              )
month_spend_average
plt.figure(figsize=(12,6))
plt.bar(month_spend_average.index,month_spend_average["TotalRevenue"])
plt.title("Month wise average revenue")
plt.xlabel("Month wise revenue")
plt.ylabel("Average Total Revenue")
plt.xticks(month_spend_average.index)
plt.tight_layout()
plt.show()

In [ ]:
#Histogram
filtered_prices = df[(df["TotalPrice"] > 0) & (df["TotalPrice"] < df["TotalPrice"].quantile(0.95))]["TotalPrice"]
plt.figure(figsize=(8,6))
plt.hist(filtered_prices, edgecolor="black")
plt.title("Distribution of Total Price")
plt.xlabel("Total Price")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
#Box plot
typical_orders = df[(df["Quantity"] > 0) & (df["Quantity"] <= 50)]["Quantity"]
plt.figure(figsize=(10,3))
plt.title("Box Plot of Order Quantities")
plt.boxplot(typical_orders)
plt.show()

###**Seaborn:**

In [ ]:
#Count plot
plt.figure(figsize=(8,6))
base_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_order = [day for day in base_order if day in df["DayName"].unique()]
sns.countplot(data=df,x="DayName",order=weekday_order)
plt.title("Sales based on days")
plt.xlabel("Days of week")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()

In [ ]:
#Violin plot
plt.figure(figsize=(8,6))
filtered_df = df[df["TotalPrice"] < df["TotalPrice"].quantile(0.95)]
sns.violinplot(data=filtered_df,x="DayType",y="TotalPrice")
plt.title("Density and Range of expense across day type")
plt.xlabel("Day Type")
plt.ylabel("Expense")
plt.tight_layout()
plt.show()

In [ ]:
#Heatmap
corr_matrix = df[anlyse_columns].corr()
sns.heatmap(corr_matrix,annot = True)
plt.title("Correlation between numerical variables")
plt.tight_layout()
plt.show()

In [ ]:
#Pair plot
#Pair plot based on customer segment
sampled_df = df[anlyse_columns + ["CustomerSegment"]].dropna().sample(
    n=min(2000, len(df)), random_state=42
)
sns.pairplot(data=sampled_df,hue="CustomerSegment")
plt.show()

###**Task 8 – Business Insights**

In [ ]:
#Identify:"Top country"
top_countries = (
    df.groupby("Country").
    agg(TotalPrice = ("TotalPrice","sum")).
    sort_values(by="TotalPrice",ascending =  False).
    reset_index()
    )
top_countries['TotalPrice'] = top_countries['TotalPrice'].round(2)
print(f'Top Country by sales is: {top_countries["Country"].iloc[0]} with sales of ${top_countries["TotalPrice"].iloc[0]}')

In [ ]:
#Identify:Best sales month
monthly_sales = df.groupby("Month")["TotalPrice"].sum().round(2)
print(f'The best sales month is: {monthly_sales.idxmax()} with sales of {monthly_sales.max()}')


In [ ]:
#Peak sales time
hourly_sales = df.groupby("Hour")["TotalPrice"].sum()
peak_hour = hourly_sales.idxmax()
print(f"The peak sales time is {peak_hour}:00 ({peak_hour % 12 or 12} {'PM' if peak_hour >= 12 else 'AM'})")

In [ ]:
#Analyze High-value customers  based on for how much value a customer has purchased
customer_spend = df.groupby("CustomerID")["TotalPrice"].sum().round(2)
customer_spend.nlargest(5)

In [ ]:
#Top products based on how many quantity is purchased
top_products = df.groupby("Description")["Quantity"].sum().round(2)
top_products.nlargest(5)